Проверка GPU

In [ ]:
!nvidia-smi

Wed Jun 24 07:24:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Подключение Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Создание папок проекта

In [ ]:
import os

PROJECT_DIR = '/content/drive/MyDrive/shelf_detection'
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)

print('Папки готовы.')
!ls -la {PROJECT_DIR}

Папки готовы.
total 12
drwx------ 2 root root 4096 Jun 23 11:25 data
drwx------ 2 root root 4096 Jun 23 11:25 models
drwx------ 2 root root 4096 Jun 23 11:25 results


Скачивание SKU-110K

In [ ]:
%cd /content/drive/MyDrive/shelf_detection/data
!pip install -q gdown
!gdown --id 1iq93lCdhaPUN0fWbLieMtzfB1850pKwd -O SKU110K_fixed.tar.gz

print('\nПроверка размера:')
!ls -lh SKU110K_fixed.tar.gz

/content/drive/MyDrive/shelf_detection/data
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1iq93lCdhaPUN0fWbLieMtzfB1850pKwd
From (redirected): https://drive.google.com/uc?id=1iq93lCdhaPUN0fWbLieMtzfB1850pKwd&confirm=t&uuid=c71cc099-b6ad-403b-a0ce-f17b1e1871a0
To: /content/drive/MyDrive/shelf_detection/data/SKU110K_fixed.tar.gz
100% 12.2G/12.2G [03:33<00:00, 57.0MB/s]

Проверка размера:
-rw------- 1 root root 12G Dec 18  2019 SKU110K_fixed.tar.gz


Распаковка на локальный диск

In [ ]:
import os
os.makedirs('/content/sku110k', exist_ok=True)

%cd /content/sku110k
!tar -xzf /content/drive/MyDrive/shelf_detection/data/SKU110K_fixed.tar.gz

print('\nГотово:')
!ls /content/sku110k/SKU110K_fixed/

/content/sku110k
^C

Готово:
annotations  images  LICENSE.txt


Создание подмножества 3000/400/1000

In [ ]:
import pandas as pd
import numpy as np
import os

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ANN_DIR = '/content/sku110k/SKU110K_fixed/annotations'
df_train_full = pd.read_csv(f'{ANN_DIR}/annotations_train.csv', header=None)
df_val_full   = pd.read_csv(f'{ANN_DIR}/annotations_val.csv',   header=None)
df_test_full  = pd.read_csv(f'{ANN_DIR}/annotations_test.csv',  header=None)

cols = ['image_name', 'x1', 'y1', 'x2', 'y2', 'class', 'img_w', 'img_h']
df_train_full.columns = cols
df_val_full.columns   = cols
df_test_full.columns  = cols

N_TRAIN, N_VAL, N_TEST = 3000, 400, 1000

train_images = pd.Series(df_train_full['image_name'].unique()).sample(n=N_TRAIN, random_state=RANDOM_SEED).tolist()
val_images   = pd.Series(df_val_full['image_name'].unique()).sample(n=N_VAL,   random_state=RANDOM_SEED).tolist()
test_images  = pd.Series(df_test_full['image_name'].unique()).sample(n=N_TEST,  random_state=RANDOM_SEED).tolist()

df_train = df_train_full[df_train_full['image_name'].isin(train_images)].reset_index(drop=True)
df_val   = df_val_full  [df_val_full  ['image_name'].isin(val_images)  ].reset_index(drop=True)
df_test  = df_test_full [df_test_full ['image_name'].isin(test_images) ].reset_index(drop=True)

SUBSET_DIR = '/content/drive/MyDrive/shelf_detection/data/subset'
os.makedirs(SUBSET_DIR, exist_ok=True)

df_train.to_csv(f'{SUBSET_DIR}/train.csv', index=False)
df_val.to_csv(  f'{SUBSET_DIR}/val.csv',   index=False)
df_test.to_csv( f'{SUBSET_DIR}/test.csv',  index=False)

print('=== Подмножество SKU-110K ===')
print(f'Train: {len(train_images)} картинок, {len(df_train)} объектов ({len(df_train)/len(train_images):.1f} среднее)')
print(f'Val:   {len(val_images)} картинок, {len(df_val)} объектов ({len(df_val)/len(val_images):.1f} среднее)')
print(f'Test:  {len(test_images)} картинок, {len(df_test)} объектов ({len(df_test)/len(test_images):.1f} среднее)')

=== Подмножество SKU-110K ===
Train: 3000 картинок, 439772 объектов (146.6 среднее)
Val:   400 картинок, 60485 объектов (151.2 среднее)
Test:  1000 картинок, 147273 объектов (147.3 среднее)


Конвертация в YOLO-формат

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

SRC_IMAGES = '/content/sku110k/SKU110K_fixed/images'
SUBSET_DIR = '/content/drive/MyDrive/shelf_detection/data/subset'
YOLO_DIR = '/content/yolo_dataset'

for split in ['train', 'val', 'test']:
    os.makedirs(f'{YOLO_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{YOLO_DIR}/labels/{split}', exist_ok=True)


def convert_split(csv_path, split_name):
    df = pd.read_csv(csv_path)
    grouped = df.groupby('image_name')
    skipped = 0
    for image_name, boxes in tqdm(grouped, desc=split_name):
        src_img = os.path.join(SRC_IMAGES, image_name)
        if not os.path.exists(src_img):
            skipped += 1
            continue
        dst_img = os.path.join(YOLO_DIR, 'images', split_name, image_name)
        if not os.path.exists(dst_img):
            os.symlink(src_img, dst_img)
        label_name = image_name.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(YOLO_DIR, 'labels', split_name, label_name)
        lines = []
        for _, row in boxes.iterrows():
            img_w, img_h = row['img_w'], row['img_h']
            x1 = max(0, min(row['x1'], img_w))
            x2 = max(0, min(row['x2'], img_w))
            y1 = max(0, min(row['y1'], img_h))
            y2 = max(0, min(row['y2'], img_h))
            if x2 <= x1 or y2 <= y1:
                continue
            x_c = (x1 + x2) / 2 / img_w
            y_c = (y1 + y2) / 2 / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            lines.append(f'0 {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}')
        with open(label_path, 'w') as f:
            f.write('\n'.join(lines))
    if skipped > 0:
        print(f'  Пропущено: {skipped}')


convert_split(f'{SUBSET_DIR}/train.csv', 'train')
convert_split(f'{SUBSET_DIR}/val.csv', 'val')
convert_split(f'{SUBSET_DIR}/test.csv', 'test')

yaml_content = """\
path: /content/yolo_dataset
train: images/train
val: images/val
test: images/test

nc: 1
names: ['product']
"""
with open(f'{YOLO_DIR}/data.yaml', 'w') as f:
    f.write(yaml_content)

print('\nГотово.')
for split in ['train', 'val', 'test']:
    n_imgs = len(os.listdir(f'{YOLO_DIR}/images/{split}'))
    n_lbls = len(os.listdir(f'{YOLO_DIR}/labels/{split}'))
    print(f'  {split}: {n_imgs} картинок, {n_lbls} лейблов')

train: 100%|██████████| 3000/3000 [00:21<00:00, 139.03it/s]


  Пропущено: 125


val: 100%|██████████| 400/400 [00:02<00:00, 135.38it/s]


  Пропущено: 18


test: 100%|██████████| 1000/1000 [00:07<00:00, 138.16it/s]

  Пропущено: 30

Готово.
  train: 2875 картинок, 2875 лейблов
  val: 382 картинок, 382 лейблов
  test: 970 картинок, 970 лейблов


Установка ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Setup complete ✅ (12 CPUs, 83.5 GB RAM, 69.6/235.7 GB disk)


Обучение YOLOv8s

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=20,
    imgsz=640,
    batch=32,
    name='yolov8s',
    project='/content/drive/MyDrive/shelf_detection/models',
    patience=10,
    save=True,
    plots=True,
    device=0,
    exist_ok=True,
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

Фиксируем результат в таблицу

In [ ]:
import pandas as pd
import os

RESULTS_FILE = '/content/drive/MyDrive/shelf_detection/results/comparison.csv'
os.makedirs('/content/drive/MyDrive/shelf_detection/results', exist_ok=True)

df = pd.DataFrame(columns=[
    'model', 'family', 'params_M', 'gflops', 'size_MB',
    'precision', 'recall', 'mAP50', 'mAP50_95',
    'inference_ms', 'fps', 'epochs', 'train_size', 'gpu'
])

# YOLOv8s
new_row = {
    'model': 'YOLOv8s',
    'family': 'Anchor-free one-stage CNN',
    'params_M': 11.13,
    'gflops': 28.4,
    'size_MB': 22.5,
    'precision': 0.904,
    'recall': 0.849,
    'mAP50': 0.892,
    'mAP50_95': 0.551,
    'inference_ms': 0.7,
    'fps': round(1000 / (0.1 + 0.7 + 1.7), 1),
    'epochs': 20,
    'train_size': 3000,
    'gpu': 'A100',
}

df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
df.to_csv(RESULTS_FILE, index=False)

print('Текущая таблица результатов:')
df

Текущая таблица результатов:


/tmp/ipykernel_974/4204180987.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


,model,family,params_M,gflops,size_MB,precision,recall,mAP50,mAP50_95,inference_ms,fps,epochs,train_size,gpu
0,YOLOv8s,Anchor-free one-stage CNN,11.13,28.4,22.5,0.904,0.849,0.892,0.551,0.7,400.0,20,3000,A100


Обучение RT-DETR

In [ ]:
# Свежий перезапуск окружения для модели
from ultralytics import RTDETR
from ultralytics.cfg import get_cfg
import yaml

# Загружаем модель
model = RTDETR('rtdetr-l.pt')

# Прибиваем imgsz в самом конфиге модели на низком уровне
model.overrides['imgsz'] = 640
if hasattr(model, 'model') and hasattr(model.model, 'args'):
    model.model.args['imgsz'] = 640

# Запускаем обучение с явным imgsz и rect=False, чтобы не было автоматического ресайза
results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=20,
    imgsz=640,
    batch=16,
    name='rtdetr_l',
    project='/content/drive/MyDrive/shelf_detection/models',
    patience=10,
    save=True,
    plots=True,
    device=0,
    exist_ok=True,
    rect=False,
    cache=False,
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr_l, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/20      33.1G      1.314     0.2664     0.4016       2274       1280: 100% ━━━━━━━━━━━━ 375/375 1.0it/s 5:60
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.2it/s 10.6s
                   all        400      60485      0.591      0.666      0.591      0.334

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/20      27.7G      0.976     0.3018     0.2241       3380       1280: 0% ──────────── 0/375  0.9s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/20      37.4G      1.179      0.274     0.2537       3390       1280: 100% ━━━━━━━━━━━━ 375/375 1.2it/s 5:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.7it/s 7.5s
                   all        400      60485      0.367       0.65      0.288      0.158

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/20      26.7G      1.525     0.2385     0.3987       3573       1280: 0% ──────────── 0/375  1.0s

...

      18/20        11G     0.5653     0.3937    0.09575        843        640: 100% ━━━━━━━━━━━━ 375/375 2.6it/s 2:26
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.9it/s 6.9s
                   all        400      60485      0.879      0.842      0.862      0.533

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      19/20        11G     0.5369     0.4224    0.08627       1004        640: 0% ──────────── 0/375  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      19/20        11G     0.5568     0.3923    0.09344        914        640: 100% ━━━━━━━━━━━━ 375/375 2.6it/s 2:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.9it/s 6.9s
                   all        400      60485      0.883      0.843      0.862      0.535

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      20/20        11G     0.3995     0.3936    0.06332       1013        640: 0% ──────────── 0/375  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      20/20        11G     0.5406     0.3923    0.08805        920        640: 100% ━━━━━━━━━━━━ 375/375 2.6it/s 2:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.9it/s 6.9s
                   all        400      60485      0.878      0.849      0.864      0.538

20 epochs completed in 1.391 hours.
Optimizer stripped from /content/drive/MyDrive/shelf_detection/models/rtdetr_l/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/shelf_detection/models/rtdetr_l/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/shelf_detection/models/rtdetr_l/weights/best.pt...
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.8s/it 23.2s
                   all        400

Обновляем таблицу результатов

In [ ]:
import pandas as pd

RESULTS_FILE = '/content/drive/MyDrive/shelf_detection/results/comparison.csv'

# Перечитываем (или создаём, если нет)
import os
if os.path.exists(RESULTS_FILE):
    df = pd.read_csv(RESULTS_FILE)
else:
    df = pd.DataFrame(columns=[
        'model', 'family', 'params_M', 'gflops', 'size_MB',
        'precision', 'recall', 'mAP50', 'mAP50_95',
        'inference_ms', 'fps', 'epochs', 'train_size', 'gpu'
    ])

# Записи. Перезапишем все известные модели свежими цифрами.
records = [
    {
        'model': 'YOLOv8s', 'family': 'Anchor-free one-stage CNN',
        'params_M': 11.13, 'gflops': 28.4, 'size_MB': 22.5,
        'precision': 0.904, 'recall': 0.849, 'mAP50': 0.892, 'mAP50_95': 0.551,
        'inference_ms': 0.7, 'fps': 400.0,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'RT-DETR-L', 'family': 'Transformer end-to-end',
        'params_M': 31.99, 'gflops': 103.4, 'size_MB': 66.2,
        'precision': 0.878, 'recall': 0.849, 'mAP50': 0.864, 'mAP50_95': 0.538,
        'inference_ms': 3.3, 'fps': 285.7,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
]

df = pd.DataFrame(records)
df.to_csv(RESULTS_FILE, index=False)
print('Текущие результаты:')
df

Текущие результаты:


,model,family,params_M,gflops,size_MB,precision,recall,mAP50,mAP50_95,inference_ms,fps,epochs,train_size,gpu
0,YOLOv8s,Anchor-free one-stage CNN,11.13,28.4,22.5,0.904,0.849,0.892,0.551,0.7,400.0,20,3000,A100
1,RT-DETR-L,Transformer end-to-end,31.99,103.4,66.2,0.878,0.849,0.864,0.538,3.3,285.7,20,3000,A100


Датасет и общие утилиты

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as F
from PIL import Image, ImageFile

# Включаем устойчивое чтение битых JPEG
ImageFile.LOAD_TRUNCATED_IMAGES = True

YOLO_DIR = '/content/yolo_dataset'


class SKU110KDataset(Dataset):
    """Читает YOLO-формат и отдаёт в torchvision-формате."""

    def __init__(self, split, img_size=640):
        self.img_dir = f'{YOLO_DIR}/images/{split}'
        self.lbl_dir = f'{YOLO_DIR}/labels/{split}'
        self.img_names = sorted([f for f in os.listdir(self.img_dir) if f.endswith('.jpg')])
        self.img_size = img_size

    def __len__(self):
        return len(self.img_names)

    def _load_one(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        lbl_path = os.path.join(self.lbl_dir, img_name.rsplit('.', 1)[0] + '.txt')

        img = Image.open(img_path)
        img.load()  # форсируем чтение, чтобы битый файл упал ЗДЕСЬ, а не позже
        img = img.convert('RGB')
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img_tensor = F.to_tensor(img)

        boxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    _, cx, cy, w, h = map(float, parts)
                    x1 = (cx - w / 2) * self.img_size
                    y1 = (cy - h / 2) * self.img_size
                    x2 = (cx + w / 2) * self.img_size
                    y2 = (cy + h / 2) * self.img_size
                    if x2 > x1 and y2 > y1:
                        boxes.append([x1, y1, x2, y2])

        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        labels = torch.ones((len(boxes),), dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
        }
        return img_tensor, target

    def __getitem__(self, idx):
        # Отказоустойчивость: если файл битый — берём следующий
        attempts = 0
        while attempts < 5:
            try:
                return self._load_one(idx)
            except (OSError, IOError) as e:
                print(f'  ! Битый файл: {self.img_names[idx]} ({e}). Беру следующий.')
                idx = (idx + 1) % len(self.img_names)
                attempts += 1
        raise RuntimeError('Не удалось прочитать ни одно изображение за 5 попыток')


def collate_fn(batch):
    return tuple(zip(*batch))


train_ds = SKU110KDataset('train', img_size=640)
val_ds   = SKU110KDataset('val',   img_size=640)
test_ds  = SKU110KDataset('test',  img_size=640)

print(f'Train: {len(train_ds)} картинок')
print(f'Val:   {len(val_ds)} картинок')
print(f'Test:  {len(test_ds)} картинок')

Train: 2875 картинок
Val:   382 картинок
Test:  970 картинок


Функция обучения

In [ ]:
import time
from tqdm import tqdm

def train_torchvision_model(model, model_name, epochs=20, batch_size=8, lr=0.005):
    """Универсальный цикл обучения для Faster R-CNN / RetinaNet."""

    device = torch.device('cuda')
    model = model.to(device)

    # Loader'ы
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, collate_fn=collate_fn, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=2, collate_fn=collate_fn, pin_memory=True)

    # Optimizer + scheduler
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    save_dir = f'/content/drive/MyDrive/shelf_detection/models/{model_name}'
    os.makedirs(save_dir, exist_ok=True)

    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        # === TRAIN ===
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs} train')
        for imgs, targets in pbar:
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(imgs, targets)
            loss = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.3f}'})

        avg_train_loss = epoch_loss / n_batches
        scheduler.step()

        # === VAL (просто прогон на инференсе для замера лосса нет — torchvision возвращает лосс только в train mode) ===
        # Поэтому здесь просто запомним loss и время эпохи
        elapsed = time.time() - start_time
        print(f'Epoch {epoch}/{epochs}: train_loss={avg_train_loss:.4f}, elapsed={elapsed/60:.1f} min')
        history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'elapsed_min': elapsed/60})

    # Сохраняем
    torch.save(model.state_dict(), f'{save_dir}/best.pt')

    # Сохраняем историю
    import pandas as pd
    pd.DataFrame(history).to_csv(f'{save_dir}/history.csv', index=False)

    total_min = (time.time() - start_time) / 60
    print(f'\n=== {model_name} обучена за {total_min:.1f} минут ===')
    print(f'Веса: {save_dir}/best.pt')
    return model

print('Функция train_torchvision_model готова.')

Функция train_torchvision_model готова.


Запуск Faster R-CNN

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# 1) Загружаем модель с предобученными весами (на COCO)
model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')

# 2) Заменяем голову классификатора под 2 класса (background + product)
num_classes = 2
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Подсчитаем параметры — для отчёта
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Faster R-CNN R50 FPN v2: {total_params:.2f} M параметров')

# 3) Обучаем
train_torchvision_model(model, model_name='faster_rcnn_r50', epochs=20, batch_size=8, lr=0.005)

Faster R-CNN R50 FPN v2: 43.26 M параметров


Epoch 1/20 train: 100%|██████████| 375/375 [02:37<00:00,  2.38it/s, loss=0.682]


Epoch 1/20: train_loss=0.9147, elapsed=2.6 min


Epoch 2/20 train: 100%|██████████| 375/375 [02:36<00:00,  2.39it/s, loss=0.773]


Epoch 2/20: train_loss=0.7004, elapsed=5.2 min


...

Epoch 19/20 train: 100%|██████████| 375/375 [02:37<00:00,  2.39it/s, loss=0.516]


Epoch 19/20: train_loss=0.4148, elapsed=49.9 min


Epoch 20/20 train: 100%|██████████| 375/375 [02:36<00:00,  2.39it/s, loss=0.356]


Epoch 20/20: train_loss=0.4142, elapsed=52.5 min

=== faster_rcnn_r50 обучена за 52.5 минут ===
Веса: /content/drive/MyDrive/shelf_detection/models/faster_rcnn_r50/best.pt


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       

Обновляем таблицу результатов

In [ ]:
import pandas as pd
import os

RESULTS_FILE = '/content/drive/MyDrive/shelf_detection/results/comparison.csv'
os.makedirs('/content/drive/MyDrive/shelf_detection/results', exist_ok=True)

records = [
    {
        'model': 'YOLOv8s', 'family': 'Anchor-free one-stage CNN',
        'params_M': 11.13, 'gflops': 28.4, 'size_MB': 22.5,
        'precision': 0.904, 'recall': 0.849, 'mAP50': 0.892, 'mAP50_95': 0.551,
        'inference_ms': 0.7, 'fps': 400.0, 'train_min': 9.7,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'RT-DETR-L', 'family': 'Transformer end-to-end',
        'params_M': 31.99, 'gflops': 103.4, 'size_MB': 66.2,
        'precision': 0.878, 'recall': 0.849, 'mAP50': 0.864, 'mAP50_95': 0.538,
        'inference_ms': 3.3, 'fps': 285.7, 'train_min': 83.5,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'Faster R-CNN R50', 'family': 'Two-stage CNN',
        'params_M': 43.26, 'gflops': None, 'size_MB': None,
        'precision': None, 'recall': None, 'mAP50': None, 'mAP50_95': None,
        'inference_ms': None, 'fps': None, 'train_min': 52.5,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
]

df = pd.DataFrame(records)
df.to_csv(RESULTS_FILE, index=False)
print('Текущая таблица результатов:')
df[['model', 'family', 'params_M', 'mAP50_95', 'train_min']]

Текущая таблица результатов:


,model,family,params_M,mAP50_95,train_min
0,YOLOv8s,Anchor-free one-stage CNN,11.13,0.551,9.7
1,RT-DETR-L,Transformer end-to-end,31.99,0.538,83.5
2,Faster R-CNN R50,Two-stage CNN,43.26,NaN,52.5


Запуск RetinaNet

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
import torch
from functools import partial

# Загружаем с предобученными весами COCO
model_retina = retinanet_resnet50_fpn_v2(weights='DEFAULT')

# Меняем голову классификатора под наши классы (background + product)
num_classes = 2
num_anchors = model_retina.head.classification_head.num_anchors
in_channels = model_retina.head.classification_head.conv[0][0].in_channels

model_retina.head.classification_head = RetinaNetClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes,
    norm_layer=partial(torch.nn.GroupNorm, 32)
)

total_params = sum(p.numel() for p in model_retina.parameters()) / 1e6
print(f'RetinaNet R50 FPN v2: {total_params:.2f} M параметров')

train_torchvision_model(
    model_retina,
    model_name='retinanet_r50',
    epochs=20,
    batch_size=8,
    lr=0.001
)

Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth


100%|██████████| 146M/146M [00:00<00:00, 183MB/s]


RetinaNet R50 FPN v2: 36.35 M параметров


Epoch 1/20 train: 100%|██████████| 360/360 [02:30<00:00,  2.40it/s, loss=0.542]


Epoch 1/20: train_loss=0.6147, elapsed=2.5 min


...

Epoch 19/20 train: 100%|██████████| 360/360 [02:36<00:00,  2.30it/s, loss=0.350]


Epoch 19/20: train_loss=0.3905, elapsed=52.4 min


Epoch 20/20 train: 100%|██████████| 360/360 [02:40<00:00,  2.24it/s, loss=0.423]


Epoch 20/20: train_loss=0.3902, elapsed=55.1 min

=== retinanet_r50 обучена за 55.1 минут ===
Веса: /content/drive/MyDrive/shelf_detection/models/retinanet_r50/best.pt


RetinaNet(
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      

Обновляем таблицу

In [ ]:
import pandas as pd
import os

RESULTS_FILE = '/content/drive/MyDrive/shelf_detection/results/comparison.csv'
os.makedirs('/content/drive/MyDrive/shelf_detection/results', exist_ok=True)

records = [
    {
        'model': 'YOLOv8s', 'family': 'Anchor-free one-stage CNN',
        'params_M': 11.13, 'gflops': 28.4, 'size_MB': 22.5,
        'precision': 0.904, 'recall': 0.849, 'mAP50': 0.892, 'mAP50_95': 0.551,
        'inference_ms': 0.7, 'fps': 400.0, 'train_min': 9.7,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'RT-DETR-L', 'family': 'Transformer end-to-end',
        'params_M': 31.99, 'gflops': 103.4, 'size_MB': 66.2,
        'precision': 0.878, 'recall': 0.849, 'mAP50': 0.864, 'mAP50_95': 0.538,
        'inference_ms': 3.3, 'fps': 285.7, 'train_min': 83.5,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'Faster R-CNN R50', 'family': 'Two-stage CNN',
        'params_M': 43.26, 'gflops': None, 'size_MB': 165.3,
        'precision': None, 'recall': None, 'mAP50': None, 'mAP50_95': None,
        'inference_ms': None, 'fps': None, 'train_min': 52.5,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'RetinaNet R50', 'family': 'Focal-loss one-stage CNN',
        'params_M': 36.35, 'gflops': None, 'size_MB': None,
        'precision': None, 'recall': None, 'mAP50': None, 'mAP50_95': None,
        'inference_ms': None, 'fps': None, 'train_min': 55.1,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
]

df = pd.DataFrame(records)
df.to_csv(RESULTS_FILE, index=False)
print('Обновлено. Текущая таблица:')
df[['model', 'family', 'params_M', 'train_min']]

Обновлено. Текущая таблица:


,model,family,params_M,train_min
0,YOLOv8s,Anchor-free one-stage CNN,11.13,9.7
1,RT-DETR-L,Transformer end-to-end,31.99,83.5
2,Faster R-CNN R50,Two-stage CNN,43.26,52.5
3,RetinaNet R50,Focal-loss one-stage CNN,36.35,55.1


Установка effdet

In [ ]:
!pip install -q effdet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 12.0 MB/s eta 0:00:00


Адаптация датасета под effdet

In [ ]:
import torch
from torch.utils.data import Dataset

class SKU110KDatasetEffDet(Dataset):
    """Обёртка SKU110KDataset под формат effdet."""

    def __init__(self, base_dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, target = self.base[idx]
        boxes_xyxy = target['boxes']  # [N, 4] в формате xyxy
        # effdet хочет yxyx (top, left, bottom, right)
        if len(boxes_xyxy) > 0:
            boxes_yxyx = boxes_xyxy[:, [1, 0, 3, 2]]
        else:
            boxes_yxyx = torch.zeros((0, 4), dtype=torch.float32)

        effdet_target = {
            'bbox': boxes_yxyx,
            'cls': target['labels'].float(),
            'img_size': torch.tensor([self.base.img_size, self.base.img_size]),
            'img_scale': torch.tensor([1.0]),
        }
        return img, effdet_target


def effdet_collate(batch):
    """Паддит боксы до одинаковой длины внутри батча."""
    imgs = torch.stack([b[0] for b in batch])
    max_n = max(len(b[1]['bbox']) for b in batch)
    max_n = max(max_n, 1)

    bboxes  = torch.zeros((len(batch), max_n, 4), dtype=torch.float32)
    classes = torch.full((len(batch), max_n), -1, dtype=torch.float32)

    for i, (_, t) in enumerate(batch):
        n = len(t['bbox'])
        if n > 0:
            bboxes[i, :n] = t['bbox']
            classes[i, :n] = t['cls']

    img_sizes  = torch.stack([b[1]['img_size'] for b in batch]).float()
    img_scales = torch.stack([b[1]['img_scale'] for b in batch]).float()

    target = {
        'bbox': bboxes,
        'cls': classes,
        'img_size': img_sizes,
        'img_scale': img_scales,
    }
    return imgs, target


train_ds_eff = SKU110KDatasetEffDet(train_ds)
val_ds_eff   = SKU110KDatasetEffDet(val_ds)

print(f'Train: {len(train_ds_eff)} картинок')
print(f'Val:   {len(val_ds_eff)} картинок')

# Быстрая проверка
img, target = train_ds_eff[0]
print(f'\nПример: картинка {img.shape}, объектов {len(target["bbox"])}')
print(f'Формат бокса yxyx: {target["bbox"][0].tolist()}')

Train: 2875 картинок
Val:   382 картинок

Пример: картинка torch.Size([3, 640, 640]), объектов 141
Формат бокса yxyx: [113.65087890625, 44.02143859863281, 172.2755126953125, 89.31231689453125]


 Создание модели EfficientDet-D0 и обучение

In [ ]:
from effdet import create_model, get_efficientdet_config, EfficientDet, DetBenchTrain, DetBenchPredict
from effdet.config.model_config import efficientdet_model_param_dict
import torch
import time
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd

# Создаём модель D0 с предобученными весами COCO
model_eff = create_model(
    'tf_efficientdet_d0',
    bench_task='train',           # обёртка для обучения (вычисляет loss)
    num_classes=1,                # только один класс
    image_size=(640, 640),
    bench_labeler=True,
    pretrained=True,
)

total_params = sum(p.numel() for p in model_eff.parameters()) / 1e6
print(f'EfficientDet-D0: {total_params:.2f} M параметров')

# Обучение
device = torch.device('cuda')
model_eff = model_eff.to(device)

EPOCHS = 20
BATCH = 16
LR = 1e-4

train_loader = DataLoader(
    train_ds_eff, batch_size=BATCH, shuffle=True,
    num_workers=2, collate_fn=effdet_collate, pin_memory=True
)

optimizer = torch.optim.AdamW(model_eff.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

save_dir = '/content/drive/MyDrive/shelf_detection/models/efficientdet_d0'
os.makedirs(save_dir, exist_ok=True)

history = []
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model_eff.train()
    epoch_loss = 0.0
    n_batches = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} train')
    for imgs, target in pbar:
        imgs = imgs.to(device)
        target = {k: v.to(device) for k, v in target.items()}

        output = model_eff(imgs, target)
        loss = output['loss']

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.3f}'})

    avg_train_loss = epoch_loss / n_batches
    scheduler.step()

    elapsed = time.time() - start_time
    print(f'Epoch {epoch}/{EPOCHS}: train_loss={avg_train_loss:.4f}, elapsed={elapsed/60:.1f} min')
    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'elapsed_min': elapsed/60})

# Сохраняем
torch.save(model_eff.state_dict(), f'{save_dir}/best.pt')
pd.DataFrame(history).to_csv(f'{save_dir}/history.csv', index=False)

total_min = (time.time() - start_time) / 60
print(f'\n=== EfficientDet-D0 обучена за {total_min:.1f} минут ===')
print(f'Веса: {save_dir}/best.pt')

Downloading: "https://github.com/rwightman/efficientdet-pytorch/releases/download/v0.1/tf_efficientdet_d0_34-f153e0cf.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientdet_d0_34-f153e0cf.pth
EfficientDet-D0: 3.83 M параметров


Epoch 1/20 train: 100%|██████████| 180/180 [02:38<00:00,  1.14it/s, loss=0.620]


Epoch 1/20: train_loss=0.9840, elapsed=2.6 min


Epoch 2/20 train: 100%|██████████| 180/180 [02:51<00:00,  1.05it/s, loss=0.439]


Epoch 2/20: train_loss=0.5115, elapsed=5.5 min


...

Epoch 18/20: train_loss=0.3055, elapsed=50.5 min


Epoch 19/20 train: 100%|██████████| 180/180 [02:58<00:00,  1.01it/s, loss=0.305]


Epoch 19/20: train_loss=0.3050, elapsed=53.4 min


Epoch 20/20 train: 100%|██████████| 180/180 [02:53<00:00,  1.04it/s, loss=0.306]

Epoch 20/20: train_loss=0.3042, elapsed=56.3 min

=== EfficientDet-D0 обучена за 56.3 минут ===
Веса: /content/drive/MyDrive/shelf_detection/models/efficientdet_d0/best.pt


In [ ]:
import pandas as pd
import os

RESULTS_FILE = '/content/drive/MyDrive/shelf_detection/results/comparison.csv'

records = [
    {
        'model': 'YOLOv8s', 'family': 'Anchor-free one-stage CNN',
        'params_M': 11.13, 'gflops': 28.4, 'size_MB': 22.5,
        'precision': 0.904, 'recall': 0.849, 'mAP50': 0.892, 'mAP50_95': 0.551,
        'inference_ms': 0.7, 'fps': 400.0, 'train_min': 9.7,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'RT-DETR-L', 'family': 'Transformer end-to-end',
        'params_M': 31.99, 'gflops': 103.4, 'size_MB': 66.2,
        'precision': 0.878, 'recall': 0.849, 'mAP50': 0.864, 'mAP50_95': 0.538,
        'inference_ms': 3.3, 'fps': 285.7, 'train_min': 83.5,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'Faster R-CNN R50', 'family': 'Two-stage CNN',
        'params_M': 43.26, 'gflops': None, 'size_MB': 165.3,
        'precision': None, 'recall': None, 'mAP50': None, 'mAP50_95': None,
        'inference_ms': None, 'fps': None, 'train_min': 52.5,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'RetinaNet R50', 'family': 'Focal-loss one-stage CNN',
        'params_M': 36.35, 'gflops': None, 'size_MB': None,
        'precision': None, 'recall': None, 'mAP50': None, 'mAP50_95': None,
        'inference_ms': None, 'fps': None, 'train_min': 55.1,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'EfficientDet-D0', 'family': 'Compound-scaled CNN',
        'params_M': 3.83, 'gflops': None, 'size_MB': None,
        'precision': None, 'recall': None, 'mAP50': None, 'mAP50_95': None,
        'inference_ms': None, 'fps': None, 'train_min': 56.3,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
]

df = pd.DataFrame(records)
df.to_csv(RESULTS_FILE, index=False)
print('Промежуточная таблица (без метрик качества — заполнится после оценки):')
df[['model', 'family', 'params_M', 'train_min']]

Промежуточная таблица (без метрик качества — заполнится после оценки):


,model,family,params_M,train_min
0,YOLOv8s,Anchor-free one-stage CNN,11.13,9.7
1,RT-DETR-L,Transformer end-to-end,31.99,83.5
2,Faster R-CNN R50,Two-stage CNN,43.26,52.5
3,RetinaNet R50,Focal-loss one-stage CNN,36.35,55.1
4,EfficientDet-D0,Compound-scaled CNN,3.83,56.3


Установка pycocotools и общие утилиты

In [ ]:
!pip install -q pycocotools

import numpy as np
import torch
import os
import json
import time
from PIL import Image, ImageFile
from torchvision.transforms import functional as F
ImageFile.LOAD_TRUNCATED_IMAGES = True

YOLO_DIR = '/content/yolo_dataset'
TEST_IMG_DIR = f'{YOLO_DIR}/images/test'
TEST_LBL_DIR = f'{YOLO_DIR}/labels/test'

# Получаем список всех картинок в test
TEST_IMAGES = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.jpg')])
print(f'Тестовых картинок: {len(TEST_IMAGES)}')


def load_gt_for_image(img_name, img_size=640):
    """Загружает ground truth для одной картинки в формате [x1, y1, x2, y2]."""
    label_path = os.path.join(TEST_LBL_DIR, img_name.rsplit('.', 1)[0] + '.txt')
    boxes = []
    if not os.path.exists(label_path):
        return np.zeros((0, 4), dtype=np.float32)
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            _, cx, cy, w, h = map(float, parts)
            x1 = (cx - w/2) * img_size
            y1 = (cy - h/2) * img_size
            x2 = (cx + w/2) * img_size
            y2 = (cy + h/2) * img_size
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
    return np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)


def compute_iou_matrix(boxes_a, boxes_b):
    """Матрица IoU всех пар (a, b)."""
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)))
    a = boxes_a[:, None]  # [N, 1, 4]
    b = boxes_b[None, :]  # [1, M, 4]
    x1 = np.maximum(a[..., 0], b[..., 0])
    y1 = np.maximum(a[..., 1], b[..., 1])
    x2 = np.minimum(a[..., 2], b[..., 2])
    y2 = np.minimum(a[..., 3], b[..., 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    area_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    union = area_a + area_b - inter
    return inter / np.maximum(union, 1e-9)


def match_predictions(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5):
    """Жадный матчинг по IoU. Возвращает TP, FP, FN."""
    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes)
    if len(gt_boxes) == 0:
        return 0, len(pred_boxes), 0
    iou = compute_iou_matrix(pred_boxes, gt_boxes)
    # Сортируем предсказания по убыванию score
    order = np.argsort(-pred_scores)
    matched_gt = set()
    tp = 0
    for pi in order:
        # Лучший gt для этого предсказания, если ещё свободен
        best_gt = -1
        best_iou = iou_thresh
        for gi in range(len(gt_boxes)):
            if gi in matched_gt:
                continue
            if iou[pi, gi] > best_iou:
                best_iou = iou[pi, gi]
                best_gt = gi
        if best_gt >= 0:
            tp += 1
            matched_gt.add(best_gt)
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - len(matched_gt)
    return tp, fp, fn


print('Утилиты загружены.')

Тестовых картинок: 970
Утилиты загружены.


Расчёт mAP через pycocotools

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def compute_map_coco(all_predictions, img_size=640):
    """
    all_predictions: словарь {img_name: {'boxes': [[x1,y1,x2,y2],...], 'scores': [...]}}
    Возвращает mAP@0.5 и mAP@0.5:0.95.
    """
    # Строим COCO-структуру ground truth
    images = []
    annotations = []
    ann_id = 0
    img_id_map = {}

    for i, img_name in enumerate(TEST_IMAGES):
        img_id_map[img_name] = i
        images.append({
            'id': i,
            'file_name': img_name,
            'width': img_size,
            'height': img_size,
        })
        gt_boxes = load_gt_for_image(img_name, img_size)
        for box in gt_boxes:
            x1, y1, x2, y2 = box
            annotations.append({
                'id': ann_id,
                'image_id': i,
                'category_id': 1,
                'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],  # COCO: xywh
                'area': float((x2 - x1) * (y2 - y1)),
                'iscrowd': 0,
            })
            ann_id += 1

    coco_gt_dict = {
        'images': images,
        'annotations': annotations,
        'categories': [{'id': 1, 'name': 'product'}],
    }

    # Сохраняем во временный файл (требование pycocotools)
    with open('/tmp/coco_gt.json', 'w') as f:
        json.dump(coco_gt_dict, f)
    coco_gt = COCO('/tmp/coco_gt.json')

    # Строим предсказания в формате COCO
    coco_dets = []
    for img_name, pred in all_predictions.items():
        if img_name not in img_id_map:
            continue
        i_id = img_id_map[img_name]
        for box, score in zip(pred['boxes'], pred['scores']):
            x1, y1, x2, y2 = box
            coco_dets.append({
                'image_id': i_id,
                'category_id': 1,
                'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                'score': float(score),
            })

    if not coco_dets:
        return {'mAP50': 0.0, 'mAP50_95': 0.0, 'precision': 0.0, 'recall': 0.0}

    coco_dt = coco_gt.loadRes(coco_dets)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    # Извлекаем precision/recall на IoU=0.5
    # Метрики coco_eval.stats: [AP@.5:.95, AP@.5, AP@.75, AP_small, AP_med, AP_large, AR_1, AR_10, AR_100, AR_small, AR_med, AR_large]
    return {
        'mAP50': float(coco_eval.stats[1]),
        'mAP50_95': float(coco_eval.stats[0]),
        'AR_100': float(coco_eval.stats[8]),  # recall с лимитом 100 детекций
    }


print('Функция compute_map_coco готова.')

Функция compute_map_coco готова.


Универсальный инференсер
Принимает модель + название фреймворка, возвращает предсказания.

In [ ]:
def run_inference(model, framework, batch_size=8, conf_thresh=0.25, device='cuda', img_size=640, max_dets=300):
    predictions = {}
    times_ms = []
    device_t = torch.device(device)

    if framework in ('ultralytics_yolo', 'ultralytics_rtdetr'):
        for img_name in tqdm(TEST_IMAGES, desc=framework):
            img_path = os.path.join(TEST_IMG_DIR, img_name)
            t0 = time.time()
            try:
                res = model(img_path, conf=conf_thresh, imgsz=img_size, verbose=False, max_det=max_dets)
            except Exception as e:
                print(f'  ! Ошибка на {img_name}: {e}')
                predictions[img_name] = {'boxes': np.zeros((0, 4), dtype=np.float32),
                                          'scores': np.zeros(0, dtype=np.float32)}
                continue
            torch.cuda.synchronize()
            elapsed = (time.time() - t0) * 1000
            times_ms.append(elapsed)

            # Защита от пустого результата (битый JPEG)
            if not res or len(res) == 0:
                predictions[img_name] = {'boxes': np.zeros((0, 4), dtype=np.float32),
                                          'scores': np.zeros(0, dtype=np.float32)}
                continue

            r = res[0]
            if r.boxes is None or len(r.boxes) == 0:
                predictions[img_name] = {'boxes': np.zeros((0, 4), dtype=np.float32),
                                          'scores': np.zeros(0, dtype=np.float32)}
                continue

            orig_h, orig_w = r.orig_shape
            boxes = r.boxes.xyxy.cpu().numpy()
            scores = r.boxes.conf.cpu().numpy()
            sx = img_size / orig_w
            sy = img_size / orig_h
            boxes = boxes * np.array([sx, sy, sx, sy])
            predictions[img_name] = {'boxes': boxes, 'scores': scores}

    elif framework == 'torchvision':
        model.eval().to(device_t)
        for img_name in tqdm(TEST_IMAGES, desc=framework):
            img_path = os.path.join(TEST_IMG_DIR, img_name)
            try:
                img = Image.open(img_path).convert('RGB').resize((img_size, img_size), Image.BILINEAR)
            except Exception as e:
                print(f'  ! Ошибка чтения {img_name}: {e}')
                predictions[img_name] = {'boxes': np.zeros((0, 4), dtype=np.float32),
                                          'scores': np.zeros(0, dtype=np.float32)}
                continue
            img_tensor = F.to_tensor(img).to(device_t)

            torch.cuda.synchronize()
            t0 = time.time()
            with torch.no_grad():
                out = model([img_tensor])[0]
            torch.cuda.synchronize()
            elapsed = (time.time() - t0) * 1000
            times_ms.append(elapsed)

            mask = out['scores'] > conf_thresh
            boxes = out['boxes'][mask].cpu().numpy()
            scores = out['scores'][mask].cpu().numpy()
            predictions[img_name] = {'boxes': boxes, 'scores': scores}

    elif framework == 'effdet':
        model.eval().to(device_t)
        for img_name in tqdm(TEST_IMAGES, desc=framework):
            img_path = os.path.join(TEST_IMG_DIR, img_name)
            try:
                img = Image.open(img_path).convert('RGB').resize((img_size, img_size), Image.BILINEAR)
            except Exception as e:
                print(f'  ! Ошибка чтения {img_name}: {e}')
                predictions[img_name] = {'boxes': np.zeros((0, 4), dtype=np.float32),
                                          'scores': np.zeros(0, dtype=np.float32)}
                continue
            img_tensor = F.to_tensor(img).unsqueeze(0).to(device_t)

            torch.cuda.synchronize()
            t0 = time.time()
            with torch.no_grad():
                out = model(img_tensor)
            torch.cuda.synchronize()
            elapsed = (time.time() - t0) * 1000
            times_ms.append(elapsed)

            out = out[0].cpu().numpy()
            mask = out[:, 4] > conf_thresh
            boxes = out[mask, :4]
            scores = out[mask, 4]
            predictions[img_name] = {'boxes': boxes, 'scores': scores}

    avg_ms = float(np.mean(times_ms)) if times_ms else 0.0
    return predictions, avg_ms

print('run_inference обновлён (с защитой от битых JPEG).')

run_inference обновлён (с защитой от битых JPEG).


Расчёт MAE/RMSE и сводный отчёт

In [ ]:
def compute_counting_metrics(predictions, conf_thresh=0.25):
    """MAE и RMSE по числу товаров на картинку."""
    diffs = []
    for img_name in TEST_IMAGES:
        gt = load_gt_for_image(img_name)
        pred = predictions.get(img_name, {'boxes': np.zeros((0,4)), 'scores': np.zeros(0)})
        pred_count = int(np.sum(pred['scores'] >= conf_thresh))
        gt_count = len(gt)
        diffs.append(pred_count - gt_count)
    diffs = np.array(diffs, dtype=np.float32)
    mae = float(np.mean(np.abs(diffs)))
    rmse = float(np.sqrt(np.mean(diffs**2)))
    return mae, rmse


def aggregate_tpfpfn(predictions, iou_thresh=0.5):
    """Сумма TP/FP/FN по всему тесту."""
    tot_tp = tot_fp = tot_fn = 0
    for img_name in TEST_IMAGES:
        gt = load_gt_for_image(img_name)
        pred = predictions.get(img_name, {'boxes': np.zeros((0,4)), 'scores': np.zeros(0)})
        tp, fp, fn = match_predictions(pred['boxes'], pred['scores'], gt, iou_thresh)
        tot_tp += tp; tot_fp += fp; tot_fn += fn
    prec = tot_tp / (tot_tp + tot_fp + 1e-9)
    rec  = tot_tp / (tot_tp + tot_fn + 1e-9)
    return prec, rec, tot_tp, tot_fp, tot_fn


print('Функции метрик готовы.')

Функции метрик готовы.


Оценка YOLOv8s

In [ ]:
from ultralytics import YOLO
from tqdm import tqdm

print('=== YOLOv8s ===')
model_yolo = YOLO('/content/drive/MyDrive/shelf_detection/models/yolov8s/weights/best.pt')
preds_yolo, ms_yolo = run_inference(model_yolo, framework='ultralytics_yolo')
metrics_yolo = compute_map_coco(preds_yolo)
mae_yolo, rmse_yolo = compute_counting_metrics(preds_yolo)
prec_yolo, rec_yolo, tp_y, fp_y, fn_y = aggregate_tpfpfn(preds_yolo)

result_yolo = {
    'model': 'YOLOv8s',
    'mAP50': round(metrics_yolo['mAP50'], 4),
    'mAP50_95': round(metrics_yolo['mAP50_95'], 4),
    'precision': round(prec_yolo, 4),
    'recall': round(rec_yolo, 4),
    'MAE': round(mae_yolo, 2),
    'RMSE': round(rmse_yolo, 2),
    'inference_ms': round(ms_yolo, 2),
    'fps': round(1000 / ms_yolo, 1),
    'TP': tp_y, 'FP': fp_y, 'FN': fn_y,
}
print(result_yolo)

# Сохраняем сырые предсказания (пригодятся для визуализации)
import pickle
with open('/content/drive/MyDrive/shelf_detection/results/preds_yolov8s.pkl', 'wb') as f:
    pickle.dump(preds_yolo, f)

=== YOLOv8s ===


ultralytics_yolo:  66%|██████▌   | 636/970 [00:32<00:16, 19.99it/s]

WARNING ⚠️ Image Read Error /content/yolo_dataset/images/test/test_2698.jpg


ultralytics_yolo: 100%|██████████| 970/970 [00:48<00:00, 19.92it/s]


loading annotations into memory...
Done (t=0.51s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.53s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=205.25s).
Accumulating evaluation results...
DONE (t=0.81s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.411
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.625
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.483
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.280
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.548
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.407
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.005
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.052
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

Оценка RT-DETR-L

In [ ]:
from ultralytics import RTDETR

print('=== RT-DETR-L ===')
model_rtdetr = RTDETR('/content/drive/MyDrive/shelf_detection/models/rtdetr_l/weights/best.pt')
preds_rtdetr, ms_rtdetr = run_inference(model_rtdetr, framework='ultralytics_rtdetr')
metrics_rtdetr = compute_map_coco(preds_rtdetr)
mae_rtdetr, rmse_rtdetr = compute_counting_metrics(preds_rtdetr)
prec_rtdetr, rec_rtdetr, tp_r, fp_r, fn_r = aggregate_tpfpfn(preds_rtdetr)

result_rtdetr = {
    'model': 'RT-DETR-L',
    'mAP50': round(metrics_rtdetr['mAP50'], 4),
    'mAP50_95': round(metrics_rtdetr['mAP50_95'], 4),
    'precision': round(prec_rtdetr, 4),
    'recall': round(rec_rtdetr, 4),
    'MAE': round(mae_rtdetr, 2),
    'RMSE': round(rmse_rtdetr, 2),
    'inference_ms': round(ms_rtdetr, 2),
    'fps': round(1000 / ms_rtdetr, 1),
    'TP': tp_r, 'FP': fp_r, 'FN': fn_r,
}
print(result_rtdetr)

with open('/content/drive/MyDrive/shelf_detection/results/preds_rtdetr.pkl', 'wb') as f:
    pickle.dump(preds_rtdetr, f)

=== RT-DETR-L ===


ultralytics_rtdetr:  65%|██████▌   | 635/970 [00:54<00:27, 11.98it/s]

WARNING ⚠️ Image Read Error /content/yolo_dataset/images/test/test_2698.jpg


ultralytics_rtdetr: 100%|██████████| 970/970 [01:22<00:00, 11.78it/s]


loading annotations into memory...
Done (t=0.51s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.63s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=206.10s).
Accumulating evaluation results...
DONE (t=0.81s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.409
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.620
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.482
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.273
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.549
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.414
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.005
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.052
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

Оценка Faster R-CNN

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

print('=== Faster R-CNN R50 ===')
model_frcnn = fasterrcnn_resnet50_fpn_v2(weights=None)
in_features = model_frcnn.roi_heads.box_predictor.cls_score.in_features
model_frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
model_frcnn.load_state_dict(torch.load('/content/drive/MyDrive/shelf_detection/models/faster_rcnn_r50/best.pt'))

preds_frcnn, ms_frcnn = run_inference(model_frcnn, framework='torchvision')
metrics_frcnn = compute_map_coco(preds_frcnn)
mae_frcnn, rmse_frcnn = compute_counting_metrics(preds_frcnn)
prec_f, rec_f, tp_f, fp_f, fn_f = aggregate_tpfpfn(preds_frcnn)

result_frcnn = {
    'model': 'Faster R-CNN R50',
    'mAP50': round(metrics_frcnn['mAP50'], 4),
    'mAP50_95': round(metrics_frcnn['mAP50_95'], 4),
    'precision': round(prec_f, 4),
    'recall': round(rec_f, 4),
    'MAE': round(mae_frcnn, 2),
    'RMSE': round(rmse_frcnn, 2),
    'inference_ms': round(ms_frcnn, 2),
    'fps': round(1000 / ms_frcnn, 1),
    'TP': tp_f, 'FP': fp_f, 'FN': fn_f,
}
print(result_frcnn)

with open('/content/drive/MyDrive/shelf_detection/results/preds_frcnn.pkl', 'wb') as f:
    pickle.dump(preds_frcnn, f)

=== Faster R-CNN R50 ===


torchvision: 100%|██████████| 970/970 [02:04<00:00,  7.76it/s]


loading annotations into memory...
Done (t=0.51s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.48s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=205.88s).
Accumulating evaluation results...
DONE (t=0.83s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.384
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.625
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.435
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.258
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.515
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.405
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.005
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.049
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

Оценка RetinaNet

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from functools import partial

print('=== RetinaNet R50 ===')
model_retina_eval = retinanet_resnet50_fpn_v2(weights=None)
num_anchors = model_retina_eval.head.classification_head.num_anchors
in_channels = model_retina_eval.head.classification_head.conv[0][0].in_channels
model_retina_eval.head.classification_head = RetinaNetClassificationHead(
    in_channels=in_channels, num_anchors=num_anchors, num_classes=2,
    norm_layer=partial(torch.nn.GroupNorm, 32)
)
model_retina_eval.load_state_dict(torch.load('/content/drive/MyDrive/shelf_detection/models/retinanet_r50/best.pt'))

preds_retina, ms_retina = run_inference(model_retina_eval, framework='torchvision')
metrics_retina = compute_map_coco(preds_retina)
mae_retina, rmse_retina = compute_counting_metrics(preds_retina)
prec_rn, rec_rn, tp_rn, fp_rn, fn_rn = aggregate_tpfpfn(preds_retina)

result_retina = {
    'model': 'RetinaNet R50',
    'mAP50': round(metrics_retina['mAP50'], 4),
    'mAP50_95': round(metrics_retina['mAP50_95'], 4),
    'precision': round(prec_rn, 4),
    'recall': round(rec_rn, 4),
    'MAE': round(mae_retina, 2),
    'RMSE': round(rmse_retina, 2),
    'inference_ms': round(ms_retina, 2),
    'fps': round(1000 / ms_retina, 1),
    'TP': tp_rn, 'FP': fp_rn, 'FN': fn_rn,
}
print(result_retina)

with open('/content/drive/MyDrive/shelf_detection/results/preds_retina.pkl', 'wb') as f:
    pickle.dump(preds_retina, f)

=== RetinaNet R50 ===


torchvision: 100%|██████████| 970/970 [02:18<00:00,  7.02it/s]


loading annotations into memory...
Done (t=0.53s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.53s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=207.75s).
Accumulating evaluation results...
DONE (t=0.83s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.329
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.581
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.340
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.203
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.462
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.177
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.005
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.047
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

Оценка EfficientDet-D0

EfficientDet требует особой настройки — нужен DetBenchPredict для инференса (а не DetBenchTrain, как при обучении).

In [ ]:
from effdet import create_model

print('=== EfficientDet-D0 ===')
# Создаём модель в режиме инференса
model_eff_eval = create_model(
    'tf_efficientdet_d0',
    bench_task='predict',
    num_classes=1,
    image_size=(640, 640),
    pretrained=False,
)
# Загружаем веса
state = torch.load('/content/drive/MyDrive/shelf_detection/models/efficientdet_d0/best.pt')
# state — это веса DetBenchTrain, у которого внутри есть `model.` префикс
# При создании DetBenchPredict префикс тоже `model.`, должно подойти напрямую
try:
    model_eff_eval.load_state_dict(state, strict=False)
    print('Веса загружены.')
except Exception as e:
    print(f'Ошибка загрузки весов: {e}')
    raise

preds_eff, ms_eff = run_inference(model_eff_eval, framework='effdet')
metrics_eff = compute_map_coco(preds_eff)
mae_eff, rmse_eff = compute_counting_metrics(preds_eff)
prec_e, rec_e, tp_e, fp_e, fn_e = aggregate_tpfpfn(preds_eff)

result_eff = {
    'model': 'EfficientDet-D0',
    'mAP50': round(metrics_eff['mAP50'], 4),
    'mAP50_95': round(metrics_eff['mAP50_95'], 4),
    'precision': round(prec_e, 4),
    'recall': round(rec_e, 4),
    'MAE': round(mae_eff, 2),
    'RMSE': round(rmse_eff, 2),
    'inference_ms': round(ms_eff, 2),
    'fps': round(1000 / ms_eff, 1),
    'TP': tp_e, 'FP': fp_e, 'FN': fn_e,
}
print(result_eff)

with open('/content/drive/MyDrive/shelf_detection/results/preds_eff.pkl', 'wb') as f:
    pickle.dump(preds_eff, f)

=== EfficientDet-D0 ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Веса загружены.


effdet: 100%|██████████| 970/970 [02:22<00:00,  6.79it/s]


loading annotations into memory...
Done (t=0.52s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.49s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=204.01s).
Accumulating evaluation results...
DONE (t=0.82s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.355
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.590
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.395
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.197
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.519
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.408
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.005
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.048
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

Заполним итоговую таблицу для отчёта

In [ ]:
import pandas as pd
import os

RESULTS_FILE = '/content/drive/MyDrive/shelf_detection/results/comparison.csv'

records = [
    {
        'model': 'YOLOv8s', 'family': 'Anchor-free one-stage CNN',
        'params_M': 11.13, 'size_MB': 22.5, 'train_min': 9.7,
        'mAP50': 0.6251, 'mAP50_95': 0.4107,
        'precision': 0.8404, 'recall': 0.8837,
        'MAE': 16.31, 'RMSE': 26.11,
        'inference_ms': 49.57, 'fps': 20.2,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'RT-DETR-L', 'family': 'Transformer end-to-end',
        'params_M': 31.99, 'size_MB': 66.2, 'train_min': 83.5,
        'mAP50': 0.6201, 'mAP50_95': 0.4090,
        'precision': 0.6975, 'recall': 0.9106,
        'MAE': 47.17, 'RMSE': 57.28,
        'inference_ms': 83.0, 'fps': 12.0,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'Faster R-CNN R50', 'family': 'Two-stage CNN',
        'params_M': 43.26, 'size_MB': 165.3, 'train_min': 52.5,
        'mAP50': 0.6253, 'mAP50_95': 0.3835,
        'precision': 0.9386, 'recall': 0.6365,
        'MAE': 48.54, 'RMSE': 63.72,
        'inference_ms': 27.48, 'fps': 36.4,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'RetinaNet R50', 'family': 'Focal-loss one-stage CNN',
        'params_M': 36.35, 'size_MB': None, 'train_min': 55.1,
        'mAP50': 0.5809, 'mAP50_95': 0.3293,
        'precision': 0.7710, 'recall': 0.6982,
        'MAE': 25.19, 'RMSE': 39.23,
        'inference_ms': 25.21, 'fps': 39.7,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
    {
        'model': 'EfficientDet-D0', 'family': 'Compound-scaled CNN',
        'params_M': 3.83, 'size_MB': None, 'train_min': 56.3,
        'mAP50': 0.5900, 'mAP50_95': 0.3547,
        'precision': 0.8993, 'recall': 0.6041,
        'MAE': 49.33, 'RMSE': 64.67,
        'inference_ms': 32.83, 'fps': 30.5,
        'epochs': 20, 'train_size': 3000, 'gpu': 'A100',
    },
]

df = pd.DataFrame(records)
df.to_csv(RESULTS_FILE, index=False)
print('Итоговая таблица сохранена.')
df

Итоговая таблица сохранена.


,model,family,params_M,size_MB,train_min,mAP50,mAP50_95,precision,recall,MAE,RMSE,inference_ms,fps,epochs,train_size,gpu
0,YOLOv8s,Anchor-free one-stage CNN,11.13,22.5,9.7,0.6251,0.4107,0.8404,0.8837,16.31,26.11,49.57,20.2,20,3000,A100
1,RT-DETR-L,Transformer end-to-end,31.99,66.2,83.5,0.6201,0.4090,0.6975,0.9106,47.17,57.28,83.00,12.0,20,3000,A100
2,Faster R-CNN R50,Two-stage CNN,43.26,165.3,52.5,0.6253,0.3835,0.9386,0.6365,48.54,63.72,27.48,36.4,20,3000,A100
3,RetinaNet R50,Focal-loss one-stage CNN,36.35,NaN,55.1,0.5809,0.3293,0.7710,0.6982,25.19,39.23,25.21,39.7,20,3000,A100
4,EfficientDet-D0,Compound-scaled CNN,3.83,NaN,56.3,0.5900,0.3547,0.8993,0.6041,49.33,64.67,32.83,30.5,20,3000,A100


Добор размеров моделей

In [ ]:
import os
import pandas as pd

RESULTS_FILE = '/content/drive/MyDrive/shelf_detection/results/comparison.csv'
df = pd.read_csv(RESULTS_FILE)

paths = {
    'YOLOv8s': '/content/drive/MyDrive/shelf_detection/models/yolov8s/weights/best.pt',
    'RT-DETR-L': '/content/drive/MyDrive/shelf_detection/models/rtdetr_l/weights/best.pt',
    'Faster R-CNN R50': '/content/drive/MyDrive/shelf_detection/models/faster_rcnn_r50/best.pt',
    'RetinaNet R50': '/content/drive/MyDrive/shelf_detection/models/retinanet_r50/best.pt',
    'EfficientDet-D0': '/content/drive/MyDrive/shelf_detection/models/efficientdet_d0/best.pt',
}

for model_name, path in paths.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        df.loc[df['model'] == model_name, 'size_MB'] = round(size_mb, 1)
        print(f'{model_name}: {size_mb:.1f} МБ')
    else:
        print(f'{model_name}: НЕ НАЙДЕН')

df.to_csv(RESULTS_FILE, index=False)
print('\nОбновлённая таблица:')
df[['model', 'params_M', 'size_MB', 'mAP50_95', 'MAE', 'fps']]

YOLOv8s: 22.5 МБ
RT-DETR-L: 66.2 МБ
Faster R-CNN R50: 173.4 МБ
RetinaNet R50: 145.7 МБ
EfficientDet-D0: 17.0 МБ

Обновлённая таблица:


,model,params_M,size_MB,mAP50_95,MAE,fps
0,YOLOv8s,11.13,22.5,0.4107,16.31,20.2
1,RT-DETR-L,31.99,66.2,0.4090,47.17,12.0
2,Faster R-CNN R50,43.26,173.4,0.3835,48.54,36.4
3,RetinaNet R50,36.35,145.7,0.3293,25.19,39.7
4,EfficientDet-D0,3.83,17.0,0.3547,49.33,30.5


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pickle
import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Загружаем сохранённые предсказания всех моделей
print('Загрузка предсказаний...')
with open('/content/drive/MyDrive/shelf_detection/results/preds_yolov8s.pkl', 'rb') as f:
    preds_all = {'YOLOv8s': pickle.load(f)}
with open('/content/drive/MyDrive/shelf_detection/results/preds_rtdetr.pkl', 'rb') as f:
    preds_all['RT-DETR-L'] = pickle.load(f)
with open('/content/drive/MyDrive/shelf_detection/results/preds_frcnn.pkl', 'rb') as f:
    preds_all['Faster R-CNN R50'] = pickle.load(f)
with open('/content/drive/MyDrive/shelf_detection/results/preds_retina.pkl', 'rb') as f:
    preds_all['RetinaNet R50'] = pickle.load(f)
with open('/content/drive/MyDrive/shelf_detection/results/preds_eff.pkl', 'rb') as f:
    preds_all['EfficientDet-D0'] = pickle.load(f)


def visualize_tp_fp_fn(img_name, preds_by_model, conf_thresh=0.25, iou_thresh=0.5, img_size=640):
    """Строит мозаику 2x3: оригинал + 5 моделей."""
    img_path = f'/content/yolo_dataset/images/test/{img_name}'
    img = Image.open(img_path).convert('RGB').resize((img_size, img_size), Image.BILINEAR)
    gt = load_gt_for_image(img_name, img_size)

    fig, axes = plt.subplots(2, 3, figsize=(24, 18))
    axes = axes.flatten()

    # Первый subplot — оригинал с GT
    axes[0].imshow(img)
    for box in gt:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.2,
                                  edgecolor='blue', facecolor='none')
        axes[0].add_patch(rect)
    axes[0].set_title(f'Ground Truth: {len(gt)} товаров', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # 5 моделей
    model_names = list(preds_by_model.keys())
    for i, model_name in enumerate(model_names, start=1):
        ax = axes[i]
        ax.imshow(img)

        pred = preds_by_model[model_name].get(img_name, {'boxes': np.zeros((0,4)), 'scores': np.zeros(0)})
        mask = pred['scores'] >= conf_thresh
        pred_boxes = pred['boxes'][mask]
        pred_scores = pred['scores'][mask]

        # Матчим предсказания с GT
        tp_boxes, fp_boxes = [], []
        matched_gt = set()
        if len(pred_boxes) > 0 and len(gt) > 0:
            iou_mat = compute_iou_matrix(pred_boxes, gt)
            order = np.argsort(-pred_scores)
            for pi in order:
                best_gt, best_iou = -1, iou_thresh
                for gi in range(len(gt)):
                    if gi in matched_gt:
                        continue
                    if iou_mat[pi, gi] > best_iou:
                        best_iou = iou_mat[pi, gi]
                        best_gt = gi
                if best_gt >= 0:
                    tp_boxes.append(pred_boxes[pi])
                    matched_gt.add(best_gt)
                else:
                    fp_boxes.append(pred_boxes[pi])
        else:
            fp_boxes = list(pred_boxes)

        fn_boxes = [gt[gi] for gi in range(len(gt)) if gi not in matched_gt]

        # Рисуем
        for box in tp_boxes:
            x1, y1, x2, y2 = box
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                                            edgecolor='lime', facecolor='none'))
        for box in fp_boxes:
            x1, y1, x2, y2 = box
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                                            edgecolor='red', facecolor='none'))
        for box in fn_boxes:
            x1, y1, x2, y2 = box
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                                            edgecolor='yellow', facecolor='none', linestyle='--'))

        title = (f'{model_name}\n'
                 f'TP={len(tp_boxes)}  FP={len(fp_boxes)}  FN={len(fn_boxes)}\n'
                 f'Найдено: {len(pred_boxes)} (правда: {len(gt)})')
        ax.set_title(title, fontsize=12)
        ax.axis('off')

    plt.suptitle(f'Сравнение детекторов на {img_name}\n'
                 '🟢 TP (правильно)  🔴 FP (лишнее)  🟡 FN (пропуск, пунктир)',
                 fontsize=16, y=1.0)
    plt.tight_layout()
    return fig


# Выберем 3 показательных тестовых картинки
sample_images = ['test_0.jpg', 'test_100.jpg', 'test_500.jpg']
sample_images = [img for img in sample_images if img in TEST_IMAGES]

os.makedirs('/content/drive/MyDrive/shelf_detection/results/visualizations', exist_ok=True)

for img_name in sample_images:
    print(f'\nВизуализирую {img_name}...')
    fig = visualize_tp_fp_fn(img_name, preds_all)
    save_path = f'/content/drive/MyDrive/shelf_detection/results/visualizations/compare_{img_name.replace(".jpg", ".png")}'
    fig.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'  Сохранено: {save_path}')

print('\nГотово! Визуализации сохранены в /content/drive/MyDrive/shelf_detection/results/visualizations/')

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
def visualize_tp_fp_fn_clean(img_name, preds_by_model, conf_thresh=0.25, iou_thresh=0.5, img_size=640):
    """Версия без эмодзи + с легендой через цветные квадраты."""
    img_path = f'/content/yolo_dataset/images/test/{img_name}'
    img = Image.open(img_path).convert('RGB').resize((img_size, img_size), Image.BILINEAR)
    gt = load_gt_for_image(img_name, img_size)

    fig, axes = plt.subplots(2, 3, figsize=(24, 18))
    axes = axes.flatten()

    # GT
    axes[0].imshow(img)
    for box in gt:
        x1, y1, x2, y2 = box
        axes[0].add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.2,
                                              edgecolor='blue', facecolor='none'))
    axes[0].set_title(f'Ground Truth: {len(gt)} товаров', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    for i, model_name in enumerate(preds_by_model.keys(), start=1):
        ax = axes[i]
        ax.imshow(img)
        pred = preds_by_model[model_name].get(img_name, {'boxes': np.zeros((0,4)), 'scores': np.zeros(0)})
        mask = pred['scores'] >= conf_thresh
        pred_boxes = pred['boxes'][mask]
        pred_scores = pred['scores'][mask]

        tp_boxes, fp_boxes = [], []
        matched_gt = set()
        if len(pred_boxes) > 0 and len(gt) > 0:
            iou_mat = compute_iou_matrix(pred_boxes, gt)
            order = np.argsort(-pred_scores)
            for pi in order:
                best_gt, best_iou = -1, iou_thresh
                for gi in range(len(gt)):
                    if gi in matched_gt:
                        continue
                    if iou_mat[pi, gi] > best_iou:
                        best_iou = iou_mat[pi, gi]
                        best_gt = gi
                if best_gt >= 0:
                    tp_boxes.append(pred_boxes[pi])
                    matched_gt.add(best_gt)
                else:
                    fp_boxes.append(pred_boxes[pi])
        else:
            fp_boxes = list(pred_boxes)
        fn_boxes = [gt[gi] for gi in range(len(gt)) if gi not in matched_gt]

        for box in tp_boxes:
            x1, y1, x2, y2 = box
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                                            edgecolor='lime', facecolor='none'))
        for box in fp_boxes:
            x1, y1, x2, y2 = box
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                                            edgecolor='red', facecolor='none'))
        for box in fn_boxes:
            x1, y1, x2, y2 = box
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                                            edgecolor='yellow', facecolor='none', linestyle='--'))

        title = (f'{model_name}\n'
                 f'TP={len(tp_boxes)}  FP={len(fp_boxes)}  FN={len(fn_boxes)}\n'
                 f'Найдено: {len(pred_boxes)}  (правда: {len(gt)})')
        ax.set_title(title, fontsize=12)
        ax.axis('off')

    # Легенда
    legend_handles = [
        patches.Patch(edgecolor='blue', facecolor='none', label='Ground Truth (эталон)'),
        patches.Patch(edgecolor='lime', facecolor='none', label='TP — правильно найдено'),
        patches.Patch(edgecolor='red', facecolor='none', label='FP — ложное срабатывание'),
        patches.Patch(edgecolor='yellow', facecolor='none', linestyle='--', label='FN — пропуск (эталон не найден)'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=4, fontsize=13, frameon=False,
               bbox_to_anchor=(0.5, -0.02))

    plt.suptitle(f'Сравнение детекторов на изображении {img_name}', fontsize=16, y=1.0)
    plt.tight_layout()
    return fig


# Перегенерируем визуализации
for img_name in sample_images:
    print(f'Визуализирую {img_name}...')
    fig = visualize_tp_fp_fn_clean(img_name, preds_all)
    save_path = f'/content/drive/MyDrive/shelf_detection/results/visualizations/compare_clean_{img_name.replace(".jpg", ".png")}'
    fig.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'  Сохранено: {save_path}')

Output hidden; open in https://colab.research.google.com to view.

Финальная программа по лучшей модели

Установка Gradio

In [ ]:
!pip install -q gradio

Класс детектора + функции подсчёта

In [ ]:
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import numpy as np
import time
import os

class ShelfProductDetector:
    """
    Финальный детектор товаров на полке.
    Использует YOLOv8s — лучшую модель по нашему сравнительному анализу.
    """

    def __init__(self, weights_path='/content/drive/MyDrive/shelf_detection/models/yolov8s/weights/best.pt'):
        self.model = YOLO(weights_path)
        self.model_name = 'YOLOv8s'

    def detect(self, image, conf_thresh=0.25, img_size=640):
        """
        Детекция товаров на одном изображении.
        Возвращает: (boxes, scores, count, inference_time_ms)
        """
        t0 = time.time()
        results = self.model(image, conf=conf_thresh, imgsz=img_size, verbose=False, max_det=300)
        elapsed_ms = (time.time() - t0) * 1000

        if not results or results[0].boxes is None or len(results[0].boxes) == 0:
            return np.zeros((0, 4)), np.zeros(0), 0, elapsed_ms

        r = results[0]
        boxes = r.boxes.xyxy.cpu().numpy()
        scores = r.boxes.conf.cpu().numpy()
        return boxes, scores, len(boxes), elapsed_ms

    def draw_detections(self, image, boxes, scores, count):
        """Рисует все детекции зелёным с подписью кол-ва."""
        img = image.copy()
        draw = ImageDraw.Draw(img)
        for box in boxes:
            x1, y1, x2, y2 = box
            draw.rectangle([x1, y1, x2, y2], outline='lime', width=2)

        # Подпись с количеством
        try:
            font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 28)
        except:
            font = ImageFont.load_default()

        text = f'Найдено товаров: {count}'
        # Фон под текстом
        bbox = draw.textbbox((10, 10), text, font=font)
        draw.rectangle([bbox[0]-5, bbox[1]-5, bbox[2]+5, bbox[3]+5], fill='black')
        draw.text((10, 10), text, fill='lime', font=font)

        return img

    def evaluate_with_gt(self, image, boxes, scores, gt_boxes, iou_thresh=0.5):
        """Сравнивает с эталоном. Рисует TP/FP/FN разными цветами."""
        img = image.copy()
        draw = ImageDraw.Draw(img)

        tp_boxes, fp_boxes = [], []
        matched_gt = set()
        if len(boxes) > 0 and len(gt_boxes) > 0:
            iou_mat = compute_iou_matrix(boxes, gt_boxes)
            order = np.argsort(-scores)
            for pi in order:
                best_gt, best_iou = -1, iou_thresh
                for gi in range(len(gt_boxes)):
                    if gi in matched_gt:
                        continue
                    if iou_mat[pi, gi] > best_iou:
                        best_iou = iou_mat[pi, gi]
                        best_gt = gi
                if best_gt >= 0:
                    tp_boxes.append(boxes[pi])
                    matched_gt.add(best_gt)
                else:
                    fp_boxes.append(boxes[pi])
        else:
            fp_boxes = list(boxes)
        fn_boxes = [gt_boxes[gi] for gi in range(len(gt_boxes)) if gi not in matched_gt]

        for box in tp_boxes:
            x1, y1, x2, y2 = box
            draw.rectangle([x1, y1, x2, y2], outline='lime', width=2)
        for box in fp_boxes:
            x1, y1, x2, y2 = box
            draw.rectangle([x1, y1, x2, y2], outline='red', width=2)
        for box in fn_boxes:
            x1, y1, x2, y2 = box
            draw.rectangle([x1, y1, x2, y2], outline='yellow', width=3)

        # Подпись со статистикой
        try:
            font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 22)
        except:
            font = ImageFont.load_default()

        accuracy = len(tp_boxes) / max(len(gt_boxes), 1) * 100
        text_lines = [
            f'Эталон: {len(gt_boxes)}  Найдено: {len(boxes)}',
            f'TP={len(tp_boxes)}  FP={len(fp_boxes)}  FN={len(fn_boxes)}',
            f'Точность подсчёта: {accuracy:.1f}%',
        ]
        y_pos = 10
        for line in text_lines:
            bbox = draw.textbbox((10, y_pos), line, font=font)
            draw.rectangle([bbox[0]-5, bbox[1]-2, bbox[2]+5, bbox[3]+2], fill='black')
            draw.text((10, y_pos), line, fill='white', font=font)
            y_pos += 30

        return img, len(tp_boxes), len(fp_boxes), len(fn_boxes), accuracy


detector = ShelfProductDetector()
print(f'Финальный детектор готов: {detector.model_name}')

Финальный детектор готов: YOLOv8s


Gradio-приложение с двумя режимами

In [ ]:
import gradio as gr

def mode_inference(image, conf_thresh):
    """Режим 1: детекция + подсчёт на произвольной картинке."""
    if image is None:
        return None, "Загрузите изображение"

    pil_img = image.resize((640, 640), Image.BILINEAR)
    boxes, scores, count, ms = detector.detect(pil_img, conf_thresh=conf_thresh)
    result_img = detector.draw_detections(pil_img, boxes, scores, count)

    report = (
        f'**Найдено товаров:** {count}\n\n'
        f'**Время инференса:** {ms:.1f} мс\n\n'
        f'**Модель:** {detector.model_name} (наилучшая по нашему сравнительному анализу 5 архитектур)\n\n'
        f'**Порог уверенности:** {conf_thresh}'
    )
    return result_img, report


def mode_evaluation(image, conf_thresh):
    """Режим 2: оценка относительно эталона (для картинок из test)."""
    if image is None:
        return None, "Загрузите изображение"

    # Попробуем найти GT по имени файла. Имя файла нам не пришло — ищем сравнением пикселей сложновато.
    # Альтернатива: ищем GT для всех test картинок, выбираем по совпадению размера?
    # Простое решение: пользователь указывает имя файла отдельно.
    return None, "Используйте поле ниже для выбора тестовой картинки"


def mode_evaluation_by_name(img_name, conf_thresh):
    """Режим 2: оценка тестовой картинки по имени."""
    if not img_name:
        return None, "Выберите имя файла"

    img_path = f'/content/yolo_dataset/images/test/{img_name}'
    if not os.path.exists(img_path):
        return None, f"Файл {img_name} не найден в test"

    pil_img = Image.open(img_path).convert('RGB').resize((640, 640), Image.BILINEAR)
    gt = load_gt_for_image(img_name, img_size=640)

    boxes, scores, count, ms = detector.detect(pil_img, conf_thresh=conf_thresh)
    result_img, tp, fp, fn, acc = detector.evaluate_with_gt(pil_img, boxes, scores, gt)

    report = (
        f'**Файл:** {img_name}\n\n'
        f'**Эталон (ground truth):** {len(gt)} товаров\n\n'
        f'**Модель нашла:** {count} товаров\n\n'
        f'**TP (правильно):** {tp}\n\n'
        f'**FP (ложные срабатывания):** {fp}\n\n'
        f'**FN (пропуски):** {fn}\n\n'
        f'**Точность подсчёта:** {acc:.1f}%\n\n'
        f'**Время инференса:** {ms:.1f} мс'
    )
    return result_img, report


# Список тестовых файлов для выпадающего меню
test_files_list = sorted(os.listdir('/content/yolo_dataset/images/test'))[:50]  # первые 50

with gr.Blocks(title='Подсчёт товаров на полке') as demo:
    gr.Markdown('# Система автоматического подсчёта товаров на полке\n'
                'Реализована на основе YOLOv8s — лучшей модели по итогам '
                'сравнительного анализа 5 архитектур.')

    with gr.Tab('Режим инференса (произвольное фото)'):
        with gr.Row():
            with gr.Column():
                inp_img = gr.Image(type='pil', label='Загрузить фото полки')
                inp_conf = gr.Slider(0.1, 0.9, value=0.25, step=0.05, label='Порог уверенности')
                btn_run = gr.Button('Найти и посчитать', variant='primary')
            with gr.Column():
                out_img = gr.Image(type='pil', label='Результат')
                out_text = gr.Markdown()
        btn_run.click(mode_inference, inputs=[inp_img, inp_conf], outputs=[out_img, out_text])

    with gr.Tab('Режим оценки (тестовое фото с эталоном)'):
        gr.Markdown('Выберите тестовое изображение из SKU-110K. '
                    'Программа сравнит предсказания модели с эталонной разметкой '
                    'и подсветит ошибки.')
        with gr.Row():
            with gr.Column():
                eval_name = gr.Dropdown(choices=test_files_list, label='Тестовое изображение')
                eval_conf = gr.Slider(0.1, 0.9, value=0.25, step=0.05, label='Порог уверенности')
                btn_eval = gr.Button('Оценить', variant='primary')
            with gr.Column():
                out_eval_img = gr.Image(type='pil', label='Результат с TP/FP/FN')
                out_eval_text = gr.Markdown()
        btn_eval.click(mode_evaluation_by_name, inputs=[eval_name, eval_conf],
                       outputs=[out_eval_img, out_eval_text])

    gr.Markdown('---\n'
                '🟢 **Зелёный** — TP (правильно)  '
                '🔴 **Красный** — FP (ложное срабатывание)  '
                '🟡 **Жёлтый** — FN (пропуск)')

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://11004359ab4c9e9c27.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
